# Домашняя работа: Беллмановские обновления в MountainCar и Acrobot

Этот ноутбук знакомит вас с двумя средами классического управления: `MountainCar-v0` и `Acrobot-v1`.
Задача — реализовать табличный Q-learning с беллмановскими обновлениями, сравнить стратегии обучения и сделать выводы по каждой среде.


## Учебные цели
- разработать дискретизаторы для разных непрерывных пространств состояний (2D у `MountainCar`, 6D у `Acrobot`)
- реализовать табличный Q-learning, параметризованный спецификацией среды
- сравнить влияние дискретизации и расписаний `epsilon` в двух независимых экспериментах
- сформулировать рекомендации по настройке беллмановских алгоритмов для новых сред


## Формат работы
- Выполняйте ноутбук сверху вниз; ячейки с `TODO` должны быть заполнены кодом или текстом.
- Если запускаете ноутбук в Colab, сначала выполните установку зависимостей (ячейка ниже).
- Фиксируйте все ключевые наблюдения: графики, таблицы, числовые метрики.
- В конце заполните секции с выводами и ответами на вопросы.



### Рекомендуемый порядок работы:
1. Сначала реализуйте класс `Discretizer` — это основа для всего остального
2. Протестируйте дискретизатор на простых примерах (создайте тестовые ячейки)
3. Реализуйте методы класса `QLearningAgent` по очереди
4. Запустите небольшое обучение (100-200 эпизодов) для отладки
5. Только после этого запускайте полные эксперименты

### Установка зависимостей
Запустите ячейку ниже только в окружениях без предустановленных библиотек.


In [ ]:
# Если работаете в Colab, раскомментируйте строки ниже.
# !pip install gymnasium numpy matplotlib tqdm -q


In [ ]:
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm


In [ ]:
SEED = 2025
random.seed(SEED)
np.random.seed(SEED)


## 1A. Разведка среды `MountainCar-v0`
Перед дискретизацией исследуйте пространство состояний и динамику наград. Соберите несколько эпизодов со случайной
политикой, зафиксируйте min/max по каждой координате и опишите, какие состояния посещаются чаще.


In [ ]:
# Исследование среды MountainCar-v0
# Цель: понять диапазоны состояний для правильной дискретизации

mc_env = gym.make('MountainCar-v0')
mc_rollouts, mc_rewards, mc_lengths = [], [], []

# Собираем 10 эпизодов со случайной политикой
for ep in range(10):
    state, _ = mc_env.reset(seed=SEED + ep)
    done = False
    total_reward = 0.0
    steps = 0
    
    while not done:
        # Сохраняем текущее состояние для анализа
        mc_rollouts.append(state)
        
        # TODO: выберите случайное действие из пространства действий среды
        # Подсказка: используйте mc_env.action_space.sample()
        action = ...  # ← ваш код
        
        # Выполняем шаг в среде
        state, reward, terminated, truncated, _ = mc_env.step(action)
        
        # TODO: обновите счётчики
        # total_reward += ...
        # steps += ...
        # done = ... or ...
        
    mc_rewards.append(total_reward)
    mc_lengths.append(steps)

mc_env.close()

# Анализ собранных данных
mc_rollouts = np.asarray(mc_rollouts)
position = mc_rollouts[:, 0]  # Первая координата — позиция машины
velocity = mc_rollouts[:, 1]  # Вторая координата — скорость

print(f'Позиция: min={position.min():.3f}, max={position.max():.3f}')
print(f'Скорость: min={velocity.min():.3f}, max={velocity.max():.3f}')
print(f'Средняя награда случайной политики: {np.mean(mc_rewards):.1f}')
print(f'Средняя длина эпизода: {np.mean(mc_lengths):.1f} шагов')

## 1B. Разведка среды `Acrobot-v1`
`Acrobot` имеет шесть признаков (cos/sin углов и угловые скорости). Исследуйте диапазоны и убедитесь,
что понимаете ограничения по скоростям. Зафиксируйте наблюдения, при которых эпизод завершается.


In [ ]:
# Исследование среды Acrobot-v1
# Acrobot — двузвенный маятник с 6 признаками:
# [cos(θ1), sin(θ1), cos(θ2), sin(θ2), θ1_dot, θ2_dot]

acro_env = gym.make('Acrobot-v1')
acro_rollouts, acro_rewards, acro_lengths = [], [], []

for ep in range(10):
    state, _ = acro_env.reset(seed=SEED + 100 + ep)
    done = False
    total_reward = 0.0
    steps = 0
    
    while not done:
        acro_rollouts.append(state)
        
        # TODO: выберите случайное действие (аналогично MountainCar)
        action = ...  # ← ваш код
        
        state, reward, terminated, truncated, _ = acro_env.step(action)
        
        # TODO: обновите счётчики (аналогично MountainCar)
        # ...
        
    acro_rewards.append(total_reward)
    acro_lengths.append(steps)

acro_env.close()

# Анализ 6-мерного пространства состояний
acro_rollouts = np.asarray(acro_rollouts)
mins = acro_rollouts.min(axis=0)
maxs = acro_rollouts.max(axis=0)

feature_names = ['cos(θ1)', 'sin(θ1)', 'cos(θ2)', 'sin(θ2)', 'θ1_dot', 'θ2_dot']
for i, (name, mn, mx) in enumerate(zip(feature_names, mins, maxs)):
    print(f'{name}: [{mn:.3f}, {mx:.3f}]')

print(f'\nСредняя награда случайной политики: {np.mean(acro_rewards):.1f}')
print(f'Средняя длина эпизода: {np.mean(acro_lengths):.1f} шагов')

## 2. Спецификации сред
Чтобы переиспользовать код, опишем каждую среду через `EnvSpec`: диапазоны наблюдений, дефолтные бины и число шагов.
Вы можете добавлять свои спецификации (например, для модифицированных карт).


In [ ]:
@dataclass
class EnvSpec:
    name: str
    observation_ranges: Tuple[Tuple[float, float], ...]
    default_bins: Tuple[int, ...]
    max_steps: int
    reward_baseline: float
    description: str


ENV_SPECS: Dict[str, EnvSpec] = {
    "MountainCar-v0": EnvSpec(
        name="MountainCar-v0",
        observation_ranges=((-1.2, 0.6), (-0.07, 0.07)),
        default_bins=(24, 24),
        max_steps=200,
        reward_baseline=-110.0,
        description="2 измерения: позиция и скорость, цель — добраться до вершины",
    ),
    "Acrobot-v1": EnvSpec(
        name="Acrobot-v1",
        observation_ranges=((-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0), (-4.0, 4.0), (-9.0, 9.0)),
        default_bins=(8, 8, 8, 8, 12, 12),
        max_steps=500,
        reward_baseline=-100.0,
        description="6 признаков: cos/sin углов и угловые скорости двух звеньев",
    ),
}

print("Доступные спецификации:")
for spec in ENV_SPECS.values():
    print(f"- {spec.name}: dims={len(spec.observation_ranges)}, default_bins={spec.default_bins}")


## 3. Универсальный дискретизатор
Реализуйте `Discretizer`, который принимает список диапазонов и соответствующее число бинов на измерение.
Класс должен уметь:
1. валидировать вход (одинаковая длина `ranges` и `bins`, бины ≥ 1);
2. клиппировать наблюдения к допустимым диапазонам;
3. переводить каждое измерение в индекс бина (`np.digitize` или ручные правила);
4. разворачивать индексы в один скаляр для обращения к Q-таблице любого размера.


In [ ]:
class Discretizer:
    """
    Преобразует непрерывное состояние в дискретный индекс.
    
    Идея: разбиваем каждое измерение на bins[i] равных интервалов.
    Например, для диапазона [-1, 1] с 4 бинами получаем интервалы:
    [-1, -0.5), [-0.5, 0), [0, 0.5), [0.5, 1]
    """

    def __init__(self, ranges: Tuple[Tuple[float, float], ...], bins: Tuple[int, ...]):
        # Валидация входных данных
        if len(ranges) != len(bins):
            raise ValueError('Длина ranges и bins должна совпадать')
        if any(b < 1 for b in bins):
            raise ValueError('Число бинов должно быть ≥ 1')
        
        self.ranges = tuple(ranges)
        self.bins = tuple(bins)
        self.edges: List[np.ndarray] = []
        
        # Предвычисляем границы бинов для каждого измерения
        for (low, high), num_bins in zip(self.ranges, self.bins):
            if num_bins == 1:
                # Один бин — границ нет
                self.edges.append(np.array([], dtype=np.float32))
            else:
                # TODO: создайте массив из (num_bins - 1) равномерно распределённых границ
                # Подсказка: np.linspace(low, high, num_bins - 1)
                # Пример: для [-1, 1] и 4 бинов → границы: [-0.5, 0.0, 0.5]
                edges = ...  # ← ваш код
                self.edges.append(edges)

    def clip(self, state: np.ndarray) -> np.ndarray:
        """Ограничивает значения состояния заданными диапазонами."""
        state = np.asarray(state, dtype=np.float32)
        clipped = []
        for value, (low, high) in zip(state, self.ranges):
            # TODO: ограничьте value диапазоном [low, high]
            # Подсказка: np.clip(value, low, high)
            clipped_value = ...  # ← ваш код
            clipped.append(float(clipped_value))
        return np.asarray(clipped, dtype=np.float32)

    def to_bin_indices(self, state: np.ndarray) -> Tuple[int, ...]:
        """Преобразует состояние в кортеж индексов бинов."""
        clipped = self.clip(state)
        indices: List[int] = []
        
        for value, edges, num_bins in zip(clipped, self.edges, self.bins):
            # TODO: найдите индекс бина для value
            # Подсказка: np.digitize(value, edges) возвращает номер бина
            # Важно: результат должен быть в диапазоне [0, num_bins - 1]
            idx = ...  # ← ваш код (используйте np.digitize)
            idx = min(max(idx, 0), num_bins - 1)  # защита от выхода за границы
            indices.append(idx)
            
        return tuple(indices)

    def flat_index(self, indices: Tuple[int, ...]) -> int:
        """Преобразует многомерный индекс в одномерный (для Q-таблицы)."""
        # TODO: используйте np.ravel_multi_index для "сплющивания" индексов
        # Пример: для bins=(3, 4) индекс (1, 2) → 1*4 + 2 = 6
        return ...  # ← ваш код

    @property
    def num_states(self) -> int:
        """Общее число дискретных состояний."""
        # TODO: верните произведение всех бинов
        # Подсказка: np.prod(self.bins)
        return ...  # ← ваш код

### Тестирование дискретизатора

После реализации `Discretizer` проверьте его работу на простом примере:

In [ ]:
# Тестирование дискретизатора
# Раскомментируйте после реализации класса Discretizer

# # Простой тест: 2D пространство с 3x4 бинами
# test_disc = Discretizer(
#     ranges=((-1.0, 1.0), (-2.0, 2.0)),
#     bins=(3, 4)
# )
# 
# # Проверка 1: общее число состояний
# print(f"Всего состояний: {test_disc.num_states}")
# assert test_disc.num_states == 12, "Ошибка: должно быть 3 * 4 = 12"
# 
# # Проверка 2: клиппинг
# state = np.array([0.5, -3.0])  # -3.0 выходит за границу [-2, 2]
# clipped = test_disc.clip(state)
# print(f"Clipped [{0.5}, {-3.0}] → {clipped}")
# assert clipped[1] == -2.0, "Ошибка: должно быть обрезано до -2.0"
# 
# # Проверка 3: индексация центра
# bin_indices = test_disc.to_bin_indices(np.array([0.0, 0.0]))
# print(f"Индексы для [0.0, 0.0]: {bin_indices}")
# # Для центра диапазона ожидаем средние индексы: (1, 2)
# 
# # Проверка 4: flat index
# flat = test_disc.flat_index((1, 2))
# print(f"Flat index для (1, 2): {flat}")
# # Для bins=(3, 4): flat = 1*4 + 2 = 6
# assert flat == 6, "Ошибка в flat_index"
# 
# print("\n✅ Discretizer работает корректно!")

## 4. Конфигурация и агент Q-learning
Теперь обобщим агента. `QLearningConfig` содержит имя среды, количество эпизодов, обучение и параметры исследования.
Агент должен уметь работать с любой спецификацией из `ENV_SPECS`, используя дефолтные бины или переопределённые в конфиге.


In [ ]:
@dataclass
class QLearningConfig:
    """Конфигурация для Q-learning агента."""
    env_name: str = "MountainCar-v0"
    num_episodes: int = 4000
    max_steps: Optional[int] = None
    learning_rate: float = 0.1       # α — скорость обучения
    discount: float = 0.99           # γ — дисконт будущих наград
    epsilon_start: float = 1.0       # начальный ε (полное исследование)
    epsilon_end: float = 0.05        # финальный ε (почти жадная политика)
    epsilon_decay_episodes: int = 2000  # за сколько эпизодов снизить ε
    bins: Optional[Tuple[int, ...]] = None
    seed: int = 42


class QLearningAgent:
    """
    Табличный Q-learning агент.
    
    Алгоритм обучения (псевдокод):
    ─────────────────────────────
    1. Инициализировать Q(s, a) = 0 для всех пар
    2. Для каждого эпизода:
       a. s ← начальное состояние
       b. Повторять (для каждого шага):
          i.   Выбрать a по ε-greedy: случайно с вер. ε, иначе argmax Q(s, ·)
          ii.  Выполнить a, получить r, s'
          iii. Обновить Q(s, a) ← Q(s, a) + α·[r + γ·max_a' Q(s', a') - Q(s, a)]
          iv.  s ← s'
       c. Уменьшить ε
    """

    def __init__(self, env: gym.Env, config: QLearningConfig):
        if config.env_name not in ENV_SPECS:
            raise ValueError(f"Неизвестная среда: {config.env_name}")
        
        self.env = env
        self.config = config
        self.spec = ENV_SPECS[config.env_name]
        self.max_steps = config.max_steps or self.spec.max_steps
        bins = config.bins or self.spec.default_bins
        
        # Создаём дискретизатор
        self.discretizer = Discretizer(self.spec.observation_ranges, bins)
        self.num_actions = env.action_space.n
        
        # Q-таблица: [num_states × num_actions]
        self.q_table = np.zeros((self.discretizer.num_states, self.num_actions), dtype=np.float32)
        self.rng = np.random.default_rng(config.seed)

    def epsilon_by_episode(self, episode: int) -> float:
        """
        Линейное уменьшение epsilon от start до end.
        
        Формула: ε = start - (start - end) × min(episode / decay_episodes, 1)
        """
        decay_steps = max(1, self.config.epsilon_decay_episodes)
        # TODO: вычислите текущий epsilon
        # frac = min(episode / decay_steps, 1.0)  # доля прогресса [0, 1]
        # epsilon = start + frac * (end - start)  # линейная интерполяция
        frac = ...  # ← ваш код
        epsilon = ...  # ← ваш код
        return epsilon

    def state_index(self, obs: np.ndarray) -> int:
        """Преобразует наблюдение среды в индекс Q-таблицы."""
        # TODO: используйте дискретизатор
        # 1. Получите индексы бинов: self.discretizer.to_bin_indices(obs)
        # 2. Преобразуйте в flat index: self.discretizer.flat_index(...)
        return ...  # ← ваш код

    def select_action(self, state_idx: int, epsilon: float) -> int:
        """
        ε-greedy выбор действия.
        
        С вероятностью ε — случайное действие (исследование)
        С вероятностью (1-ε) — лучшее действие (эксплуатация)
        """
        # TODO: реализуйте ε-greedy
        # if self.rng.random() < epsilon:
        #     return случайное действие от 0 до num_actions-1
        # else:
        #     return argmax по Q-таблице для state_idx
        ...  # ← ваш код

    def greedy_action(self, state_idx: int) -> int:
        """Жадный выбор действия (argmax Q)."""
        return int(np.argmax(self.q_table[state_idx]))

    def bellman_update(self, s_idx: int, action: int, reward: float, 
                       next_idx: int, terminated: bool) -> None:
        """
        Обновление Q-значения по формуле Беллмана.
        
        Q(s, a) ← Q(s, a) + α · [target - Q(s, a)]
        
        где target = r                           если terminated
                   = r + γ · max_a' Q(s', a')    иначе
        """
        alpha = self.config.learning_rate
        gamma = self.config.discount
        
        # TODO: вычислите target
        # if terminated:
        #     target = reward  # нет будущего — только текущая награда
        # else:
        #     target = reward + gamma * max(Q[next_idx, :])  # уравнение Беллмана
        target = ...  # ← ваш код
        
        # TODO: обновите Q-таблицу
        # td_error = target - Q[s_idx, action]
        # Q[s_idx, action] += alpha * td_error
        ...  # ← ваш код

    def train(self) -> Dict[str, List[float]]:
        """Основной цикл обучения."""
        metrics = {
            'episode_reward': [], 
            'episode_length': [], 
            'epsilon': []
        }
        
        for ep in tqdm(range(self.config.num_episodes), desc=f"Train {self.config.env_name}"):
            # Сброс среды
            obs, _ = self.env.reset()
            state_idx = self.state_index(obs)
            epsilon = self.epsilon_by_episode(ep)
            
            total_reward = 0.0
            steps = 0
            
            # Цикл по шагам эпизода
            for t in range(self.max_steps):
                # TODO: 1. Выберите действие по ε-greedy
                action = ...  # ← ваш код
                
                # TODO: 2. Выполните шаг в среде
                next_obs, reward, terminated, truncated, _ = self.env.step(action)
                next_idx = self.state_index(next_obs)
                
                # TODO: 3. Обновите Q-таблицу (Беллмановское обновление)
                # Важно: передавайте terminated, а НЕ (terminated or truncated)
                # truncated означает прерывание по времени, не терминальное состояние
                ...  # ← ваш код
                
                # Обновляем счётчики
                total_reward += reward
                state_idx = next_idx
                steps += 1
                
                if terminated or truncated:
                    break
            
            # Сохраняем метрики
            metrics['episode_reward'].append(total_reward)
            metrics['episode_length'].append(steps)
            metrics['epsilon'].append(epsilon)
            
        return metrics

    def evaluate(self, episodes: int = 5) -> Tuple[float, float]:
        """Оценка агента без обучения (жадная политика)."""
        env = gym.make(self.config.env_name)
        rewards, lengths = [], []
        
        try:
            for ep in range(episodes):
                obs, _ = env.reset(seed=self.config.seed + 10_000 + ep)
                state_idx = self.state_index(obs)
                total_reward = 0.0
                
                for t in range(self.max_steps):
                    # Жадный выбор (без исследования)
                    action = self.greedy_action(state_idx)
                    obs, reward, terminated, truncated, _ = env.step(action)
                    total_reward += reward
                    state_idx = self.state_index(obs)
                    
                    if terminated or truncated:
                        break
                
                rewards.append(total_reward)
                lengths.append(t + 1)
        finally:
            env.close()
            
        return float(np.mean(rewards)), float(np.mean(lengths))

### Проверка расписания epsilon

Перед запуском обучения полезно визуализировать, как будет меняться epsilon:

In [ ]:
# Визуализация расписания epsilon
# Раскомментируйте после реализации epsilon_by_episode

# # Создаём временный конфиг
# test_config = QLearningConfig(
#     epsilon_start=1.0,
#     epsilon_end=0.05,
#     epsilon_decay_episodes=2000,
#     num_episodes=4000
# )
# 
# # Создаём агента для проверки
# temp_env = gym.make('MountainCar-v0')
# temp_agent = QLearningAgent(temp_env, test_config)
# 
# # Строим график epsilon
# episodes = np.arange(test_config.num_episodes)
# epsilons = [temp_agent.epsilon_by_episode(ep) for ep in episodes]
# 
# plt.figure(figsize=(10, 4))
# plt.plot(episodes, epsilons, 'b-', linewidth=2)
# plt.axhline(y=test_config.epsilon_end, color='r', linestyle='--', label=f'ε_end = {test_config.epsilon_end}')
# plt.axvline(x=test_config.epsilon_decay_episodes, color='g', linestyle='--', label=f'decay = {test_config.epsilon_decay_episodes}')
# plt.xlabel('Эпизод')
# plt.ylabel('Epsilon (ε)')
# plt.title('Линейное расписание epsilon: от исследования к эксплуатации')
# plt.legend()
# plt.grid(True)
# plt.show()
# 
# temp_env.close()
# 
# # Проверка значений
# assert abs(temp_agent.epsilon_by_episode(0) - 1.0) < 0.01, "ε(0) должен быть ~1.0"
# assert abs(temp_agent.epsilon_by_episode(2000) - 0.05) < 0.01, "ε(2000) должен быть ~0.05"
# assert temp_agent.epsilon_by_episode(3000) >= 0.05, "ε не должен опускаться ниже ε_end"
# print("✅ Расписание epsilon работает корректно!")

## 5. Оценка жадной политики (отдельная функция)
Иногда удобно отделить оценочную функцию от класса. Реализуйте вспомогательную функцию, которая прогоняет N
эпизодов без `epsilon` и возвращает средние награды/длины. Используйте её в экспериментах для валидации моделей.


In [ ]:
def evaluate_policy(env_name: str, agent: QLearningAgent, episodes: int = 5) -> Tuple[float, float]:
    """
    Оценка жадной политики агента в отдельной среде.
    
    Returns:
        (средняя награда, средняя длина эпизода)
    """
    env = gym.make(env_name)
    rewards, lengths = [], []
    
    try:
        for ep in range(episodes):
            obs, _ = env.reset(seed=agent.config.seed + 20_000 + ep)
            state_idx = agent.state_index(obs)
            total_reward = 0.0
            
            for t in range(agent.max_steps):
                # TODO: выберите жадное действие и выполните шаг
                action = ...  # ← используйте agent.greedy_action(state_idx)
                obs, reward, terminated, truncated, _ = env.step(action)
                
                total_reward += reward
                state_idx = agent.state_index(obs)
                
                if terminated or truncated:
                    break
            
            rewards.append(total_reward)
            lengths.append(t + 1)
    finally:
        env.close()
    
    return float(np.mean(rewards)), float(np.mean(lengths))

In [ ]:
def moving_average(values: List[float], window: int = 100) -> np.ndarray:
    """Вычисляет скользящее среднее для сглаживания графиков."""
    arr = np.asarray(values, dtype=np.float32)
    if arr.size == 0:
        return arr
    if arr.size < window:
        return arr
    kernel = np.ones(window, dtype=np.float32) / window
    return np.convolve(arr, kernel, mode='valid')

## 6. Эксперимент A — `MountainCar`: влияние дискретизации
1. Используйте одну и ту же конфигурацию обучения, но разные сетки бинов (например, `(18, 18)` vs `(30, 30)`).
2. Зафиксируйте сиды для честного сравнения.
3. Логируйте скользящее среднее награды, длины эпизодов и итоговый greedy-score.
4. Сделайте выводы о том, сколько бинов нужно для устойчивого обучения в `MountainCar`.


In [ ]:
# Эксперимент A: Влияние дискретизации на MountainCar
# Сравниваем грубую и тонкую сетки бинов

mc_results: Dict[str, Dict[str, object]] = {}

# Конфигурации для сравнения
mc_configs = {
    '20x20 (coarse)': {
        'bins': (20, 20),
        'num_episodes': 2000,
        'epsilon_decay': 1000,
        'learning_rate': 0.2,
    },
    '30x30 (fine)': {
        'bins': (30, 30),
        'num_episodes': 3000,
        'epsilon_decay': 1500,
        'learning_rate': 0.25,
    },
}

for label, cfg in mc_configs.items():
    print(f"\n{'='*50}")
    print(f"Обучение: {label}")
    print(f"{'='*50}")
    
    # TODO: создайте среду
    env = gym.make('MountainCar-v0')
    
    # TODO: создайте конфигурацию
    # config = QLearningConfig(
    #     env_name='MountainCar-v0',
    #     bins=cfg['bins'],
    #     num_episodes=cfg['num_episodes'],
    #     epsilon_decay_episodes=cfg['epsilon_decay'],
    #     learning_rate=cfg['learning_rate'],
    #     discount=0.99,
    #     seed=SEED,
    # )
    config = ...  # ← ваш код
    
    # TODO: создайте и обучите агента
    # agent = QLearningAgent(env, config)
    # logs = agent.train()
    agent = ...  # ← ваш код
    logs = ...   # ← ваш код
    
    # TODO: оцените агента
    # eval_reward, eval_length = agent.evaluate(episodes=10)
    eval_reward, eval_length = ...  # ← ваш код
    
    # Сохраняем результаты
    mc_results[label] = {
        'logs': logs,
        'eval_reward': eval_reward,
        'eval_length': eval_length,
        'bins': cfg['bins'],
    }
    
    env.close()
    n_states = cfg['bins'][0] * cfg['bins'][1]
    print(f"Результат: eval={eval_reward:.1f}, length={eval_length:.1f}, states={n_states}")

In [ ]:
# Визуализация результатов эксперимента A

if mc_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for label, result in mc_results.items():
        rewards = result['logs']['episode_reward']
        lengths = result['logs']['episode_length']
        
        # TODO: вычислите скользящее среднее для сглаживания
        # ma_rewards = moving_average(rewards, window=50)
        # ma_lengths = moving_average(lengths, window=50)
        ma_rewards = ...  # ← ваш код
        ma_lengths = ...  # ← ваш код
        
        # TODO: постройте графики
        # axes[0].plot(ma_rewards, label=f"{label} (eval {result['eval_reward']:.0f})")
        # axes[1].plot(ma_lengths, label=label)
        ...  # ← ваш код
    
    # Настройка графиков
    axes[0].set_title('MountainCar: награда (MA 50)')
    axes[0].set_xlabel('Эпизоды')
    axes[0].set_ylabel('Награда')
    axes[0].grid(True)
    axes[0].legend()
    
    axes[1].set_title('MountainCar: длина эпизода (MA 50)')
    axes[1].set_xlabel('Эпизоды')
    axes[1].set_ylabel('Шаги')
    axes[1].grid(True)
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print('Сначала запустите ячейку с экспериментом A.')

### Выводы по эксперименту A

**Ответьте на следующие вопросы:**

1. **Скорость сходимости:** Какая сетка бинов научилась быстрее? Почему?
   - TODO: ваш ответ

2. **Финальная производительность:** Какая конфигурация достигла лучшей итоговой награды при оценке?
   - TODO: ваш ответ

3. **Стабильность:** Какая сетка показала меньше колебаний в награде на поздних этапах обучения?
   - TODO: ваш ответ

4. **Рекомендации:** Какое количество бинов вы бы порекомендовали для MountainCar и почему?
   - TODO: ваш ответ

## 7. Эксперимент B — `Acrobot`: расписания epsilon и стабильность
1. Зафиксируйте одну дискретизацию (например, дефолтную) и сравните как минимум два расписания `epsilon`
   (быстрое и медленное уменьшение).
2. Проанализируйте, как часто агент достигает целевой высоты и какие награды получает.
3. Сравните variance наград и сделайте выводы, какая стратегия исследования лучше для `Acrobot`.


In [ ]:
# Эксперимент B: Влияние расписания epsilon на Acrobot
# Сравниваем быстрое и медленное уменьшение epsilon

acrobot_results: Dict[str, Dict[str, object]] = {}

# Расписания для сравнения
acrobot_schedules = {
    'fast_decay (ε→0.05 за 600 эп.)': {
        'decay': 600,
        'num_episodes': 5000,
    },
    'slow_decay (ε→0.05 за 2500 эп.)': {
        'decay': 2500,
        'num_episodes': 5000,
    },
}

for label, cfg in acrobot_schedules.items():
    print(f"\n{'='*50}")
    print(f"Обучение: {label}")
    print(f"{'='*50}")
    
    env = gym.make('Acrobot-v1')
    
    # TODO: создайте конфигурацию для Acrobot
    # Рекомендуемые параметры:
    # - learning_rate = 0.2
    # - discount = 0.995 (выше, чем для MountainCar)
    # - bins = None (используются default_bins из EnvSpec)
    config = QLearningConfig(
        env_name='Acrobot-v1',
        num_episodes=cfg['num_episodes'],
        epsilon_decay_episodes=cfg['decay'],
        learning_rate=0.2,
        discount=0.995,
        seed=SEED + 7,
    )
    
    # TODO: обучите агента
    agent = ...  # ← ваш код
    logs = ...   # ← ваш код
    
    # Оценка
    eval_reward, eval_length = agent.evaluate(episodes=10)
    
    # Подсчёт success_rate: доля эпизодов, завершившихся до max_steps
    lengths = np.asarray(logs['episode_length'])
    success_rate = float(np.mean(lengths < agent.max_steps))
    
    acrobot_results[label] = {
        'logs': logs,
        'eval_reward': eval_reward,
        'eval_length': eval_length,
        'success_rate': success_rate,
    }
    
    env.close()
    print(f"Результат: eval={eval_reward:.1f}, success_rate={success_rate:.2%}")

In [ ]:
# Визуализация результатов эксперимента B

if acrobot_results:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for label, result in acrobot_results.items():
        rewards = result['logs']['episode_reward']
        lengths = result['logs']['episode_length']
        epsilons = result['logs']['epsilon']
        
        # Используем большее окно сглаживания из-за высокой дисперсии
        ma_rewards = moving_average(rewards, window=200)
        ma_lengths = moving_average(lengths, window=200)
        
        # TODO: постройте графики для наград, длин и epsilon
        axes[0].plot(ma_rewards, label=f"{label} (eval {result['eval_reward']:.0f})")
        axes[1].plot(ma_lengths, label=label)
        axes[2].plot(epsilons, label=label)
    
    # Настройка графиков
    axes[0].set_title('Acrobot: награда (MA 200)')
    axes[0].set_xlabel('Эпизоды')
    axes[0].set_ylabel('Награда')
    axes[0].grid(True)
    axes[0].legend()
    
    axes[1].set_title('Acrobot: длина эпизода (MA 200)')
    axes[1].set_xlabel('Эпизоды')
    axes[1].set_ylabel('Шаги')
    axes[1].grid(True)
    axes[1].legend()
    
    axes[2].set_title('Расписание epsilon')
    axes[2].set_xlabel('Эпизоды')
    axes[2].set_ylabel('ε')
    axes[2].grid(True)
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()
    
    # Таблица результатов
    print("\nСводка результатов:")
    print("-" * 60)
    for label, result in acrobot_results.items():
        print(f"{label}:")
        print(f"  Eval reward: {result['eval_reward']:.1f}")
        print(f"  Success rate: {result['success_rate']:.2%}")
else:
    print('Сначала запустите ячейку с экспериментом B.')

### Выводы по эксперименту B

**Ответьте на следующие вопросы:**

1. **Влияние расписания epsilon:** Как скорость уменьшения epsilon повлияла на обучение?
   - TODO: ваш ответ

2. **Success rate:** Какое расписание привело к большей доле успешных эпизодов? Почему?
   - TODO: ваш ответ

3. **Исследование vs эксплуатация:** Объясните баланс между exploration и exploitation в каждом расписании.
   - TODO: ваш ответ

4. **Рекомендации:** Какое расписание epsilon вы бы выбрали для Acrobot при ограниченном бюджете эпизодов (например, 3000)?
   - TODO: ваш ответ

## 8. Дополнительные исследования (опционально)

Если хотите углубиться в тему, попробуйте следующие эксперименты:

### Идея 1: Адаптивная дискретизация
- Используйте неравномерную сетку бинов (больше бинов в критичных областях пространства состояний)
- Для MountainCar: больше бинов около скорости = 0 и позиции = -0.5

### Идея 2: Добавление третьей среды
- Добавьте `CartPole-v1` или `Pendulum-v1` (с дискретизацией действий)
- Создайте новый EnvSpec и проверьте универсальность вашего кода

### Идея 3: Анализ Q-таблицы
- Визуализируйте Q-values для разных состояний
- Найдите "ключевые" состояния, где агент принимает критичные решения
- Для MountainCar постройте heatmap Q(position, velocity, action)

### Идея 4: Комбинированное расписание epsilon
- Попробуйте экспоненциальное затухание вместо линейного
- Или двухфазное расписание: быстро до 0.3, затем медленно до 0.05

### Идея 5: Влияние learning rate
- Сравните разные значения α (0.05, 0.1, 0.2, 0.5)
- Постройте график зависимости скорости сходимости от α

**Ваши эксперименты:**
TODO: опишите, что вы попробовали и к каким выводам пришли

## 9. Вопросы для самопроверки

1. **TODO:** Чем отличаются требования к дискретизации для MountainCar (2D) и Acrobot (6D)? Какие измерения требуют более тонкой дискретизации и почему?

2. **TODO:** Какое расписание `epsilon` показало себя лучшим в каждой среде и почему? Есть ли универсальная стратегия или нужно адаптировать под задачу?

3. **TODO:** Какие метрики вы использовали для контроля сходимости и стабильности? Почему недостаточно смотреть только на среднюю награду?

4. **TODO:** С какими проблемами вы столкнулись при применении табличного Q-learning? Какие среды не подойдут для этого подхода?